In [1]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
import pandas as pd

train = pd.read_csv("train_cleaned.csv")

In [3]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

# Split internal: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
# Kolom kategorikal dan numerik
categorical_features = ["gender", "academic_work_impact"]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

# Preprocessing yang sama untuk seluruh model
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [5]:
# Model dan parameter dasar. ROC-AUC = 0.96051
models = {
    "HistGradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}
model = models['HistGradient Boosting']
print(model)


HistGradientBoostingClassifier(l2_regularization=1.0, learning_rate=0.08,
                               max_iter=300, random_state=42)


In [6]:
pipeline = Pipeline([
				("preprocessor", preprocessor),
				("model", model)
		])

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint


In [8]:
# Pipeline baru khusus untuk tuning
tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True
    ))
])



In [9]:
# Rentang parameter yang akan dicoba
param_distributions = {
    # Seberapa besar langkah belajar model
    "model__learning_rate": loguniform(0.02, 0.15),

    # Jumlah maksimal tahap boosting
    "model__max_iter": randint(200, 701),

    # Kompleksitas setiap pohon
    "model__max_leaf_nodes": randint(15, 64),

    # Minimum jumlah data pada daun/node akhir
    "model__min_samples_leaf": randint(10, 51),

    # Regularisasi untuk menahan overfitting
    "model__l2_regularization": loguniform(0.0001, 10.0)
}

In [10]:
# Random search: mencoba kombinasi parameter secara acak
random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=12,                 # 12 kombinasi parameter
    scoring="roc_auc",
    cv=cv,                     # StratifiedKFold 10-fold
    n_jobs=2,                  # Naikkan bila RAM/CPU aman
    verbose=2,
    random_state=42,
    refit=True,                # setelah menemukan terbaik, fit ulang pada seluruh X_train
    return_train_score=True
)

In [ ]:
# Mulai tuning
random_search.fit(X_train, y_train)

print("\nParameter terbaik:")
print(random_search.best_params_)

print(f"\nROC-AUC CV terbaik: {random_search.best_score_:.5f}")

Fitting 10 folds for each of 12 candidates, totalling 120 fits


In [ ]:
best_pipeline = random_search.best_estimator_

In [ ]:
from sklearn.metrics import roc_auc_score

y_test_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("ROC-AUC internal test:",
      roc_auc_score(y_test, y_test_prob))

In [ ]:
import pandas as pd
import joblib

pd.DataFrame(random_search.cv_results_).sort_values(
    "rank_test_score"
).to_csv("random_search_results.csv", index=False)

joblib.dump(best_pipeline, "histgradient_tuned.joblib")

print("Hasil tuning dan model terbaik berhasil disimpan.")

In [ ]:
# test = pd.read_csv('test_cleaned.csv')

# test_id = test['id']
# Xfinaltest = test.drop(columns=['id'])


In [ ]:
# # Probabilitas seseorang masuk kelas addicted_label = 1
# y_prob = pipeline.predict_proba(Xfinaltest)[:, 1]

# # Label final berdasarkan threshold default 0.5
# y_pred = pipeline.predict(Xfinaltest)

In [ ]:
# submission = pd.DataFrame({
#     "id": test_id,
#     "addicted_label": y_prob
# })

# submission.to_csv("submission.csv", index=False)